# Silver Layer: Cleaning & Feature Engineering Notebook

**Target Tables:**
- **Read:** `bronze_ad_links` (`bronze.GeneralSearch`), `bronze_extraction_data` (`bronze.InformationExtraction`)
- **Write:** `silver_clean_ads` (`silver.SilverCleanAd`)

**Objective:**
Cleans and normalizes raw Bronze listings, extracts numerical specifications (RAM, CPU, storage, screen size), applies regex feature flags (`needs_repair`, `urgent_sale`), computes one-hot encoded characteristics, and persists structured records to `silver.SilverCleanAd`.

## 1. Setup and Imports
Configure environment, regex utilities, Polars, database connection engine, and model layer modules.

In [1]:
import sys
from pathlib import Path

# Ensure project root is available in system path
project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

import re
import unicodedata

import polars as pl
from sqlalchemy import insert, select
from sqlalchemy.orm import Session

# Importing from 'app' module
from app.config import db_engine
from app.models import bronze, silver

## 2. Load Raw Bronze Listings
Query available listings from `bronze.GeneralSearch` and valid details from `bronze.InformationExtraction`.

In [2]:
with db_engine.connect() as connection:
    # Load general search links from Bronze layer
    df_general = pl.read_database(
        select(bronze.GeneralSearch).where(
            (bronze.GeneralSearch.available.is_(True))
            & (bronze.GeneralSearch.status == 200)
        ),
        connection=connection
    )

    # Load detailed extraction data from Bronze layer
    df_details = pl.read_database(
        select(bronze.InformationExtraction).where(
            (bronze.InformationExtraction.price > 50)
            & (bronze.InformationExtraction.price < 10_000)
            & (bronze.InformationExtraction.title.isnot(None))
        ),
        connection=connection
    )

## 3. Define Cleaning Utilities
Define helper functions to normalize column names by stripping accents and replacing non-alphanumeric characters.

In [3]:
def clean_col_name(col_name: str) -> str:
    """
    Normalizes column names to lowercase, ASCII underscores.
    """
    lower_name = col_name.lower()
    no_special = unicodedata.normalize('NFKD', lower_name).encode('ASCII', 'ignore').decode('utf-8')
    no_spaces = re.sub(r'[ -]', '_', no_special)
    return re.sub(r'_+', '_', no_spaces)

## 4. Feature Extraction and Engineering
Join datasets, unnest specifications, clean numerical values, apply regex defect/urgency flags, and encode product characteristics.

In [4]:
if not df_general.is_empty() and not df_details.is_empty():
    initial_full_df = (
        df_general
        .join(df_details, left_on="id", right_on="general_search_id", how="right")
        .unnest('specifications')
    )
    
    col_mapping = {col: clean_col_name(col) for col in initial_full_df.columns}
    renamed_df = initial_full_df.rename(col_mapping)
    
    needs_repair_regex = r'(?i)(defeito|detalhe|quebrado|peças|n[ãa]o liga|bateria viciada)'
    urgent_sale_regex = r'(?i)(urgente|torro|dinheiro)'
    
    col_normalized_df = (
        renamed_df
        .with_columns(
            (pl.col('para_doacao').str.to_lowercase().str.strip_chars() == 'sim').fill_null(False).alias('for_donation'),
            (pl.col('aceita_trocas').str.to_lowercase().str.strip_chars() == 'sim').fill_null(False).alias('accept_trades'),
            pl.col('datetime').dt.date().alias('date'),
            pl.col('memoria_ram').fill_null("0").str.extract(r'(\d+)').cast(pl.Int32).alias('ram_gb'),
            pl.col('armazenamento').fill_null("0").str.extract(r'(\d+)').cast(pl.Int32).alias('storage_gb'),
            pl.col('tamanho_de_tela').fill_null("0").str.extract(r'(\d+)').cast(pl.Float32).alias('screen_size_pol'),
            pl.col('title').str.replace_all(r'\s+', ' ').str.strip_chars().str.to_titlecase().alias('title'),
            pl.col('description').str.replace_all(r'\s+', ' ').str.strip_chars().fill_null("Not Informed").str.to_titlecase().alias('description'),
            pl.col('caracteristicas').str.replace_all("Inclui ", "").str.split(r', ').fill_null(["Not Informed"]).alias('characteristics'),
            pl.col('region').str.to_uppercase().alias('region'),
            pl.col('marca').fill_null("Not Informed").str.strip_chars().alias('brand'),
            pl.col('condicao').fill_null("Not Informed").str.strip_chars().alias('item_condition'),
            pl.col('marca_do_processador').fill_null("Not Informed").alias('cpu_brand'),
            pl.col('modelo_do_processador').fill_null("Not Informed").alias('cpu_model'),
            pl.col('marca_da_placa_de_video').fill_null("Not Informed").alias('gpu_brand'),
        )
        .with_columns(
            ((pl.col('description').str.contains(needs_repair_regex)) | (pl.col('title').str.contains(needs_repair_regex))).alias('needs_repair'),
            ((pl.col('description').str.contains(urgent_sale_regex)) | (pl.col('title').str.contains(urgent_sale_regex))).alias('urgent_sale')
        )
    )

## 5. Persist Clean Listings to Silver Layer (`silver_clean_ads`)
Save feature-engineered clean listings into `silver.SilverCleanAd`.

In [5]:
if 'col_normalized_df' in locals() and not col_normalized_df.is_empty():
    with Session(db_engine) as session:
        # Execute insert into Silver clean ads table
        session.execute(insert(silver.SilverCleanAd), col_normalized_df.to_dicts())
        session.commit()
        print(f"Successfully saved {len(col_normalized_df)} cleaned listings to silver_clean_ads.")
else:
    print("No clean data ready for insertion.")